# <span style="color:blue">PART 4: INTEGRATED DATA PIPELINE</span> ⏱️ 

---
In this part, we **integrate everything** learned so far — databases, APIs, and web scraping — into a **unified, production-style data pipeline**.

> 💡 **Goal:** Build a reusable `DataCollectionPipeline` class that pulls data from multiple sources, stores it in SQLite, logs every step, and exports results.

## <span style="color:blue">4.1 Complete Data Collection System</span>

Now let's **integrate everything**: databases, APIs, and web scraping!

The `DataCollectionPipeline` class provides a **unified interface** for:
- 🗄️ Querying **SQLite databases**
- 🌐 Fetching data from **REST APIs**
- 🕷️ **Web scraping** with BeautifulSoup
- 📝 **Logging** every operation to both file and console
- 📊 **Exporting** all collected data to CSV files

### Imports Overview

| Library | Purpose |
|---|---|
| `sqlite3` | Database connectivity |
| `requests` | HTTP requests for APIs and web scraping |
| `BeautifulSoup` | HTML parsing |
| `pandas` | Data manipulation and export |
| `logging` | Structured logging |
| `datetime`, `time` | Timestamps and rate limiting |
| `json` | Serializing API responses for DB storage |

In [ ]:
# ─── Core standard library imports ───────────────────────────────────────────
import sqlite3
import logging
from datetime import datetime
import time
import json
import os

# ─── Third-party imports (install via pip if needed) ─────────────────────────
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("✅ All libraries imported successfully!")

### <span style="color:blue">`DataCollectionPipeline` Class</span>

The class is organized into four method groups:
1. **`__init__` & `_create_tables`** — Setup: logging, DB connection, tables
2. **Database methods** — `collect_from_database()`
3. **API methods** — `collect_from_api()`
4. **Web scraping methods** — `collect_from_web()`
5. **Utility methods** — `_log_collection()`, `get_collection_stats()`, `export_all_data()`, `close()`

> <span style="color:red">**Important:**</span> The pipeline uses a **single SQLite database** (`collected_data.db` by default) to store data from **all sources**, keeping everything in one place for easy analysis.

In [ ]:
class DataCollectionPipeline:
    """
    Unified data collection from multiple sources.
    Integrates: SQLite databases, REST APIs, and Web Scraping.
    """

    # =========================================================================
    # INITIALIZATION
    # =========================================================================

    def __init__(self, db_path='collected_data.db'):
        """
        Initialize pipeline with database and logging.

        Args:
            db_path (str): Path to the SQLite database file.
                           Will be created if it doesn't exist.
        """
        # ── Setup logging ─────────────────────────────────────────────────────
        # Logs go to BOTH a file ('pipeline.log') and the console (StreamHandler)
        logging.basicConfig(
            level=logging.INFO,  # Log INFO and above (INFO, WARNING, ERROR, CRITICAL)
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('pipeline.log'),  # Persistent log file
                logging.StreamHandler(),  # Real-time console output
            ],
        )
        self.logger = logging.getLogger(__name__)  # Logger named after this module

        # ── Setup database ────────────────────────────────────────────────────
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path)  # Creates file if it doesn't exist
        self._create_tables()  # Ensure all tables exist

        # ── Setup HTTP session ────────────────────────────────────────────────
        # Using a Session object reuses TCP connections (faster than fresh requests)
        self.session = requests.Session()
        self.session.headers.update(
            {
                'User-Agent': 'DataPipeline/1.0 (Educational)'  # Identify our bot politely
            }
        )

        self.logger.info("Pipeline initialized")

    def _create_tables(self):
        """
        Create all required tables if they don't already exist.
        Uses CREATE TABLE IF NOT EXISTS so it's safe to call multiple times.
        """
        cursor = self.conn.cursor()

        # ── Table for API data ────────────────────────────────────────────────
        # Stores the full JSON response as TEXT (json.dumps)
        cursor.execute(
            '''
            CREATE TABLE IF NOT EXISTS api_data (
                id           INTEGER PRIMARY KEY AUTOINCREMENT,
                source       TEXT NOT NULL,       -- URL of the API endpoint
                data_type    TEXT NOT NULL,       -- e.g. 'json'
                content      TEXT NOT NULL,       -- JSON-serialized response
                collected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        '''
        )

        # ── Table for scraped data ────────────────────────────────────────────
        # Stores the URL, page title, and JSON-encoded field dict
        cursor.execute(
            '''
            CREATE TABLE IF NOT EXISTS scraped_data (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                url        TEXT NOT NULL,
                title      TEXT,
                content    TEXT,          -- JSON-encoded dict of scraped fields
                scraped_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        '''
        )

        # ── Table for pipeline operation logs ─────────────────────────────────
        # Audit trail: which source, how many records, success/error, and why
        cursor.execute(
            '''
            CREATE TABLE IF NOT EXISTS pipeline_logs (
                id               INTEGER PRIMARY KEY AUTOINCREMENT,
                source_type      TEXT NOT NULL,       -- 'database', 'api', or 'web'
                records_collected INTEGER,
                status           TEXT,               -- 'success' or 'error'
                error_message    TEXT,               -- NULL on success
                timestamp        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        '''
        )

        self.conn.commit()  # Persist table creation

    # =========================================================================
    # DATABASE METHODS
    # =========================================================================

    def collect_from_database(self, query, source_db_path):
        """
        Collect data from another SQLite database by running a SQL query.

        Args:
            query (str):           SQL SELECT query to run on the source DB.
            source_db_path (str):  Path to the source .db file.

        Returns:
            pd.DataFrame: Query results. Empty DataFrame on error.
        """
        self.logger.info(f"Collecting from database: {source_db_path}")
        try:
            # Open a separate connection to the source database
            source_conn = sqlite3.connect(source_db_path)

            # pandas read_sql_query runs the query and returns a DataFrame directly
            df = pd.read_sql_query(query, source_conn)
            source_conn.close()  # Always close the connection when done

            self.logger.info(f"✓ Collected {len(df)} records from database")
            self._log_collection('database', len(df), 'success')  # Audit log
            return df

        except Exception as e:
            # Catch-all: file not found, SQL syntax error, permission issues, etc.
            self.logger.error(f"✗ Database error: {e}")
            self._log_collection('database', 0, 'error', str(e))
            return pd.DataFrame()  # Return empty DF so callers don't need to check None

    # =========================================================================
    # API METHODS
    # =========================================================================

    def collect_from_api(self, url, params=None):
        """
        Collect JSON data from a REST API endpoint via GET request.
        The full response is serialized and stored in the 'api_data' table.

        Args:
            url (str):    Full API endpoint URL.
            params (dict): Optional query parameters (appended to URL).

        Returns:
            dict | list | None: Parsed JSON response, or None on error.
        """
        self.logger.info(f"Collecting from API: {url}")
        try:
            # timeout=10 prevents the pipeline from hanging indefinitely
            response = self.session.get(url, params=params, timeout=10)
            response.raise_for_status()  # Raises HTTPError for 4xx/5xx status codes

            data = response.json()  # Parse JSON response body

            # Persist raw JSON to the database for later analysis
            cursor = self.conn.cursor()
            cursor.execute(
                '''
                INSERT INTO api_data (source, data_type, content)
                VALUES (?, ?, ?)
            ''',
                (url, 'json', json.dumps(data)),
            )  # json.dumps converts dict → string
            self.conn.commit()

            self.logger.info(f"✓ Collected API data from {url}")
            self._log_collection('api', 1, 'success')
            return data

        except Exception as e:
            # Covers: network errors, timeouts, bad JSON, HTTP errors
            self.logger.error(f"✗ API error: {e}")
            self._log_collection('api', 0, 'error', str(e))
            return None

    # =========================================================================
    # WEB SCRAPING METHODS
    # =========================================================================

    def collect_from_web(self, url, selectors):
        """
        Scrape structured data from a webpage using CSS selectors.

        Args:
            url (str):        Page URL to scrape.
            selectors (dict): Mapping of field name → CSS selector string.

        Example selectors:
            selectors = {
                'title':       'h1.title',
                'price':       'span.price',
                'description': 'p.desc'
            }

        Returns:
            dict: {field: scraped_text}. Values are None if selector not found.
        """
        self.logger.info(f"Scraping: {url}")
        try:
            response = self.session.get(url, timeout=10)
            response.raise_for_status()

            # Parse HTML with lxml (faster than html.parser; pip install lxml)
            soup = BeautifulSoup(response.text, 'lxml')

            # Extract each field using its CSS selector
            data = {}
            for field, selector in selectors.items():
                element = soup.select_one(selector)  # Returns first match or None
                data[field] = element.get_text(strip=True) if element else None
                # strip=True removes leading/trailing whitespace from text

            # Persist to database: store full data dict as JSON in 'content' column
            cursor = self.conn.cursor()
            cursor.execute(
                '''
                INSERT INTO scraped_data (url, title, content)
                VALUES (?, ?, ?)
            ''',
                (url, data.get('title'), json.dumps(data)),
            )
            self.conn.commit()

            self.logger.info(f"✓ Scraped data from {url}")
            self._log_collection('web', 1, 'success')
            return data

        except Exception as e:
            self.logger.error(f"✗ Scraping error: {e}")
            self._log_collection('web', 0, 'error', str(e))
            return {}  # Return empty dict instead of None for safe iteration

    # =========================================================================
    # UTILITY METHODS
    # =========================================================================

    def _log_collection(self, source_type, records, status, error_msg=None):
        """
        Internal method: Write a collection event to the pipeline_logs table.
        Called automatically after every collect_* method (success or failure).

        Args:
            source_type (str): 'database', 'api', or 'web'
            records (int):     Number of records collected (0 on error)
            status (str):      'success' or 'error'
            error_msg (str):   Exception message if status=='error', else None
        """
        cursor = self.conn.cursor()
        cursor.execute(
            '''
            INSERT INTO pipeline_logs (source_type, records_collected, status, error_message)
            VALUES (?, ?, ?, ?)
        ''',
            (source_type, records, status, error_msg),
        )
        self.conn.commit()

    def get_collection_stats(self):
        """
        Query the database to summarize all data collection activity.

        Returns:
            dict with keys:
                'api_records'     (int): total rows in api_data table
                'scraped_records' (int): total rows in scraped_data table
                'logs'            (DataFrame): per-source success counts
        """
        stats = {}
        cursor = self.conn.cursor()

        # Count total API records collected
        cursor.execute("SELECT COUNT(*) FROM api_data")
        stats['api_records'] = cursor.fetchone()[0]

        # Count total scraped records collected
        cursor.execute("SELECT COUNT(*) FROM scraped_data")
        stats['scraped_records'] = cursor.fetchone()[0]

        # Summarize logs: per source_type — total attempts and how many succeeded
        logs_df = pd.read_sql_query(
            '''
            SELECT
                source_type,
                COUNT(*) AS count,
                SUM(CASE WHEN status='success' THEN 1 ELSE 0 END) AS successful
            FROM pipeline_logs
            GROUP BY source_type
        ''',
            self.conn,
        )
        stats['logs'] = logs_df

        return stats

    def export_all_data(self, output_dir='exports'):
        """
        Export all collected data tables to CSV files.
        Creates the output directory if it doesn't exist.

        Files created:
            {output_dir}/api_data.csv
            {output_dir}/scraped_data.csv
            {output_dir}/pipeline_logs.csv
        """
        os.makedirs(output_dir, exist_ok=True)  # Create dir (no error if exists)

        # Export API data
        api_df = pd.read_sql_query("SELECT * FROM api_data", self.conn)
        api_df.to_csv(f'{output_dir}/api_data.csv', index=False)

        # Export scraped data
        scraped_df = pd.read_sql_query("SELECT * FROM scraped_data", self.conn)
        scraped_df.to_csv(f'{output_dir}/scraped_data.csv', index=False)

        # Export operation logs (useful for debugging and auditing)
        logs_df = pd.read_sql_query("SELECT * FROM pipeline_logs", self.conn)
        logs_df.to_csv(f'{output_dir}/pipeline_logs.csv', index=False)

        self.logger.info(f"✓ Data exported to {output_dir}/")

    def close(self):
        """
        Close the SQLite database connection.
        Always call this when finished to avoid database locking issues.
        """
        self.conn.close()
        self.logger.info("Pipeline closed")


print("✅ DataCollectionPipeline class defined successfully!")

### Usage Example

The following code shows a **complete end-to-end run** of the pipeline:

1. **Collect from database** — query a local SQLite DB (`library.db`)
2. **Collect from API** — fetch a GitHub repository's metadata
3. **Scrape from web** — pull book details from [books.toscrape.com](http://books.toscrape.com)
4. **Get statistics** — see how many records came from each source
5. **Export everything** — save all data to CSV files
6. **Close** — cleanly shut down the pipeline

> <span style="color:red">**Note:**</span> `library.db` must exist locally for step 1 to succeed. Steps 2 and 3 require internet access.

In [ ]:
# ─── Initialize the pipeline ──────────────────────────────────────────────────
# Creates 'collected_data.db' and sets up logging
pipeline = DataCollectionPipeline()

# ─── Step 1: Collect from a local SQLite database ─────────────────────────────
# Reads books with a rating above 4 from a pre-existing library database
# (Requires 'library.db' to exist — created in Part 1 of the tutorial)
library_data = pipeline.collect_from_database(
    "SELECT * FROM books WHERE rating > 4",  # SQL query
    'library.db',  # Path to source database
)

# ─── Step 2: Collect from the GitHub REST API ─────────────────────────────────
# Fetches metadata (stars, forks, description, etc.) for the pandas repository
github_data = pipeline.collect_from_api(
    'https://api.github.com/repos/pandas-dev/pandas'
    # No authentication needed for public repos (but rate-limited to 60 req/hr)
)

# ─── Step 3: Scrape structured data from a webpage ───────────────────────────
# CSS selectors map field names to HTML elements on the target page
book_data = pipeline.collect_from_web(
    'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html',
    {
        'title': 'h1',  # Book title in <h1>
        'price': '.price_color',  # Price in element with class 'price_color'
        'availability': '.availability',  # In-stock status
        'description': '#product_description ~ p',  # Sibling <p> after #product_description
    },
)

# ─── Step 4: Print collection statistics ─────────────────────────────────────
stats = pipeline.get_collection_stats()
print("\n=== Collection Statistics ===")
print(f"API Records:     {stats['api_records']}")
print(f"Scraped Records: {stats['scraped_records']}")
print("\nLogs by Source Type:")
print(stats['logs'])

# ─── Step 5: Export all collected data to CSV ─────────────────────────────────
# Creates: exports/api_data.csv, exports/scraped_data.csv, exports/pipeline_logs.csv
pipeline.export_all_data()

# ─── Step 6: Close the pipeline ───────────────────────────────────────────────
# Always close to release the database file lock
pipeline.close()

---

## <span style="color:blue">4.2 Final Project: Book Market Intelligence System</span>

**Objective**: Create a **complete data collection system** that gathers book data from multiple sources and generates **market insights**.

---

### 📦 Data Sources *(all 3 required)*

| # | Source | Description |
|---|--------|-------------|
| 1 | **Database** | Use the library database from Part 1 |
| 2 | **API** | Use GitHub API to find book-related repositories |
| 3 | **Web** | Scrape the Books to Scrape website |

---

### 📋 Deliverables

**1. Code** (`final_project.py`):
- Complete `DataCollectionPipeline` implementation
- All three collection methods working
- Error handling and logging
- Data validation

**2. Database** (`market_intelligence.db`):
- Properly structured tables
- Collected data from all sources
- Relationships between tables

**3. Analysis Report** (`analysis.pdf` or `analysis.html`):
- Executive summary
- Data collection statistics
- Market insights:
  - Popular genres
  - Price trends
  - Rating patterns
  - Technology trends (from GitHub)
- Visualizations (**at least 5**)
- Recommendations

**4. Documentation** (`README.md`):
- Installation instructions
- How to run
- Architecture explanation
- Data schema diagram

---

### 🏆 Grading *(100 points)*

| Category | Points | Breakdown |
|---|---|---|
| **Data Collection** | 40 | Database (10) + API (15) + Web Scraping (15) |
| **Code Quality** | 25 | Clean code (10) + Error handling (8) + Logging (7) |
| **Analysis** | 25 | Insights quality (15) + Visualizations (10) |
| **Documentation** | 10 | README completeness (5) + Code comments (5) |


---

## <span style="color:blue">✅ Key Takeaways</span>

| Concept | Summary |
|---|---|
| 🗄️ **Databases** | Best for structured data, complex queries, and data integrity |
| 🌐 **APIs** | Official channels, real-time data — rate limits apply |
| 🕷️ **Web Scraping** | Last resort — be ethical, always check `robots.txt` |
| 🔗 **Integration** | Combine sources for comprehensive insights |
| ✅ **Best Practices** | Error handling, logging, validation, documentation |

---

### <span style="color:red">Remember:</span>

- Always **check if an API exists** before resorting to scraping
- **Respect rate limits** and `robots.txt`
- **Validate and clean** your data before analysis
- **Document your process** for reproducibility
- Be **ethical and legal** in all data collection

---

> 🎓 **End of Tutorial** — You now have a fully integrated, production-style data pipeline!